# Federated Transfer Learning — 2 Different Binary Medical Datasets

**What "Federated Transfer Learning" means here:** normal Federated Learning assumes every client has data
that looks roughly the same and a model built from scratch. Here instead:

1. We start from a **pretrained backbone** (ResNet-18, pretrained on ImageNet) — this is the "transfer
   learning" part: the backbone already knows general visual features (edges, textures, shapes) so each
   client doesn't need to learn vision from zero.
2. The backbone is **frozen** (shared, identical, never changes) and only a small binary classifier **head**
   is trained — this head is what gets federated (trained locally by each client, then averaged by the
   server, round after round).
3. The two clients hold **two completely different binary medical datasets** — this is the federated part:
   the raw images look nothing alike, but both problems are "is this scan normal (0) or abnormal (1)?", so a
   shared binary head is meaningful.

| Client | Dataset | Task |
|---|---|---|
| **Client 1** | **PneumoniaMNIST** (chest X-rays) | Normal (0) vs Pneumonia (1) |
| **Client 2** | **BreastMNIST** (breast ultrasound) | Benign (0) vs Malignant (1) |

Both datasets come from **[MedMNIST](https://medmnist.com/)** — small (28x28), lightweight, and already
binary, so they're ideal for a quick classroom-scale federated transfer learning demo.

---
## Section 1: Imports & Device Setup

In [1]:
import sys
!{sys.executable} -m pip install -q torch torchvision medmnist

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from medmnist import PneumoniaMNIST, BreastMNIST

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(42); torch.manual_seed(42)
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')

Device  : cpu
PyTorch : 2.11.0+cpu


---
## Section 2: Configuration

In [3]:
CONFIG = {
    'num_rounds'      : 10,
    'local_epochs'    : 3,
    'batch_size'      : 32,
    'learning_rate'   : 0.001,
    'image_size'      : 96,     # resize target so ResNet-18's pretrained features stay meaningful
    'mu_prox'         : 0.01,   # FedProx proximal strength
}
CLASS_NAMES = ['Normal / Benign (0)', 'Pneumonia / Malignant (1)']
print('Configuration loaded')
for k, v in CONFIG.items(): print(f'   {k}: {v}')

Configuration loaded
   num_rounds: 10
   local_epochs: 3
   batch_size: 32
   learning_rate: 0.001
   image_size: 96
   mu_prox: 0.01


---
## Section 3: Load the Two Binary Medical Datasets

In [4]:
# ImageNet normalization -> required because the backbone was pretrained on ImageNet
imagenet_mean, imagenet_std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

transform_med = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),      # MedMNIST images are 1-channel -> replicate to 3
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
transform_med_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class BinaryMedMNIST(Dataset):
    """Thin wrapper so every client returns (image_tensor, int_label) with label in {0, 1}."""
    def __init__(self, medmnist_dataset):
        self.ds = medmnist_dataset
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        img, label = self.ds[i]
        return img, int(label[0]) if hasattr(label, '__len__') else int(label)

# --- Client 1: PneumoniaMNIST (chest X-rays, normal vs pneumonia) ---
pneumonia_train_raw = PneumoniaMNIST(split='train', transform=transform_med,      download=True)
pneumonia_test_raw  = PneumoniaMNIST(split='test',  transform=transform_med_test, download=True)
pneumonia_train_ds  = BinaryMedMNIST(pneumonia_train_raw)
pneumonia_test_ds   = BinaryMedMNIST(pneumonia_test_raw)

# --- Client 2: BreastMNIST (breast ultrasound, benign vs malignant) ---
breast_train_raw = BreastMNIST(split='train', transform=transform_med,      download=True)
breast_test_raw  = BreastMNIST(split='test',  transform=transform_med_test, download=True)
breast_train_ds  = BinaryMedMNIST(breast_train_raw)
breast_test_ds   = BinaryMedMNIST(breast_test_raw)

client_names = ['Client 1 (PneumoniaMNIST - Chest X-ray)', 'Client 2 (BreastMNIST - Ultrasound)']
client_train_datasets = [pneumonia_train_ds, breast_train_ds]
client_test_datasets  = [pneumonia_test_ds, breast_test_ds]
client_sizes = [len(pneumonia_train_ds), len(breast_train_ds)]

client_loaders      = [DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=True) for ds in client_train_datasets]
client_test_loaders = [DataLoader(ds, batch_size=64, shuffle=False) for ds in client_test_datasets]

for name, tr, te in zip(client_names, client_train_datasets, client_test_datasets):
    print(f'   {name}: {len(tr)} train / {len(te)} test images')

   Client 1 (PneumoniaMNIST - Chest X-ray): 4708 train / 624 test images
   Client 2 (BreastMNIST - Ultrasound): 546 train / 156 test images


---
## Section 4: Pretrained Backbone (frozen) + Federated Binary Head

In [5]:
class FederatedTransferModel(nn.Module):
    """
    Transfer learning: ResNet-18 pretrained on ImageNet acts as a FROZEN feature extractor,
    shared identically by every client and never trained. Only the small binary classifier
    head on top is trainable - that head is what gets federated.
    """
    def __init__(self, freeze_backbone=True):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        backbone.fc = nn.Identity()          # strip ImageNet's 1000-class head, keep 512-d features
        self.backbone = backbone
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(512, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        with torch.set_grad_enabled(self.training and any(p.requires_grad for p in self.backbone.parameters())):
            feats = self.backbone(x)
        return self.head(feats)

_probe = FederatedTransferModel()
n_total   = sum(p.numel() for p in _probe.parameters())
n_trained = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print(f'Total parameters      : {n_total:,}')
print(f'Trainable (head only) : {n_trained:,}  <- this is what gets federated across the 2 clients')
del _probe

Total parameters      : 11,209,474
Trainable (head only) : 32,962  <- this is what gets federated across the 2 clients


---
## Section 5: Helper Functions

In [6]:
def train_local(model, loader, optimizer, epochs, device):
    model.train()
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            ce(model(imgs), labels).backward()
            optimizer.step()

def train_local_fedprox(model, loader, optimizer, epochs, device, global_state, mu):
    model.train()
    ce = nn.CrossEntropyLoss()
    global_params = {k: v.clone().detach().to(device) for k, v in global_state.items()}
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = ce(model(imgs), labels)
            prox = sum(((p - global_params[n]) ** 2).sum()
                       for n, p in model.named_parameters() if p.requires_grad and n in global_params)
            (loss + (mu / 2) * prox).backward()
            optimizer.step()

def evaluate_model(model, loader, device):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total

def get_soft_preds(model, public_X, device):
    model.eval()
    with torch.no_grad():
        return F.softmax(model(public_X.to(device)), dim=1).cpu()

def weighted_avg_head_state(models_list, sizes):
    """Only average the trainable HEAD parameters - the frozen backbone is identical across
    clients already, so there's nothing meaningful to average there."""
    total = sum(sizes)
    keys = models_list[0].head.state_dict().keys()
    return {k: sum(models_list[i].head.state_dict()[k].float() * (sizes[i] / total) for i in range(len(models_list)))
            for k in keys}

print('Helper functions ready')

Helper functions ready


---
## Section 6: Combined Evaluation Set + Run FedAvg / FedProx / FedAdam

In [ ]:
from torch.utils.data import ConcatDataset
test_loader_combined = DataLoader(ConcatDataset(client_test_datasets), batch_size=64, shuffle=False)

def run_weight_based_algo(algo_name, use_prox=False):
    label = 'FedProx' if use_prox else algo_name.upper()
    print('='*60); print(f'  {label}'); print('='*60)

    clients = [FederatedTransferModel().to(DEVICE) for _ in range(2)]
    opts    = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
               for m in clients]
    global_model = FederatedTransferModel().to(DEVICE)
    server_m, server_v = {}, {}
    history = {'round': [], 'combined_acc': [], 'acc_pneumonia': [], 'acc_breast': []}

    for rnd in range(1, CONFIG['num_rounds'] + 1):
        g_head_state = global_model.head.state_dict()
        for c in clients:
            c.head.load_state_dict(g_head_state)

        for c, opt, loader in zip(clients, opts, client_loaders):
            if use_prox:
                train_local_fedprox(c, loader, opt, CONFIG['local_epochs'], DEVICE, g_head_state, CONFIG['mu_prox'])
            else:
                train_local(c, loader, opt, CONFIG['local_epochs'], DEVICE)

        avg_head = weighted_avg_head_state(clients, client_sizes)

        if algo_name == 'fedadam':
            beta1, beta2, tau, server_lr = 0.9, 0.99, 1e-3, 0.1
            new_head = {}
            for key in avg_head:
                delta = avg_head[key].float() - g_head_state[key].float()
                if key not in server_m:
                    server_m[key] = torch.zeros_like(delta)
                    server_v[key] = torch.full_like(delta, tau ** 2)
                server_m[key] = beta1 * server_m[key] + (1 - beta1) * delta
                server_v[key] = beta2 * server_v[key] + (1 - beta2) * (delta ** 2)
                new_head[key] = g_head_state[key].float() + server_lr * server_m[key] / (torch.sqrt(server_v[key]) + tau)
            global_model.head.load_state_dict(new_head)
        else:
            global_model.head.load_state_dict(avg_head)   # plain FedAvg-style update (also used for FedProx)

        g_acc = evaluate_model(global_model, test_loader_combined, DEVICE)
        p_acc = evaluate_model(global_model, client_test_loaders[0], DEVICE)
        b_acc = evaluate_model(global_model, client_test_loaders[1], DEVICE)
        history['round'].append(rnd); history['combined_acc'].append(g_acc)
        history['acc_pneumonia'].append(p_acc); history['acc_breast'].append(b_acc)
        print(f'  Round {rnd:02d} | Combined: {g_acc:.4f} | Pneumonia: {p_acc:.4f} | Breast: {b_acc:.4f}')
    print(f'Final combined accuracy: {history["combined_acc"][-1]:.4f}\n')
    return history

# 1. FedAvg
history_fedavg  = run_weight_based_algo('fedavg')
# 2. FedProx
history_fedprox = run_weight_based_algo('fedavg', use_prox=True)
# 3. FedAdam
history_fedadam = run_weight_based_algo('fedadam')

  FEDAVG
  Round 01 | Combined: 0.5744 | Pneumonia: 0.6490 | Breast: 0.2756
  Round 02 | Combined: 0.6667 | Pneumonia: 0.7516 | Breast: 0.3269
  Round 03 | Combined: 0.6128 | Pneumonia: 0.6875 | Breast: 0.3141
  Round 04 | Combined: 0.6410 | Pneumonia: 0.7212 | Breast: 0.3205
  Round 05 | Combined: 0.6987 | Pneumonia: 0.7837 | Breast: 0.3590
  Round 06 | Combined: 0.6897 | Pneumonia: 0.7724 | Breast: 0.3590
  Round 07 | Combined: 0.6846 | Pneumonia: 0.7692 | Breast: 0.3462
  Round 08 | Combined: 0.6872 | Pneumonia: 0.7644 | Breast: 0.3782
  Round 09 | Combined: 0.7000 | Pneumonia: 0.7644 | Breast: 0.4423
  Round 10 | Combined: 0.6718 | Pneumonia: 0.7404 | Breast: 0.3974
Final combined accuracy: 0.6718

  FedProx
  Round 01 | Combined: 0.6808 | Pneumonia: 0.7772 | Breast: 0.2949
  Round 02 | Combined: 0.6449 | Pneumonia: 0.7356 | Breast: 0.2821
  Round 03 | Combined: 0.6974 | Pneumonia: 0.7885 | Breast: 0.3333
  Round 04 | Combined: 0.6833 | Pneumonia: 0.7644 | Breast: 0.3590
  Round 05

---
## Section 7: Run FedDistillation (FedDF)

In [ ]:
# 4. FedDistillation (FedDF) - clients share PREDICTIONS on a small shared public proxy set, not weights.
print('='*60); print('  FEDDISTILLATION (FedDF)'); print('='*60)

# Public proxy set: a small pooled sample from both clients' test data
public_pool   = ConcatDataset([torch.utils.data.Subset(pneumonia_test_ds, range(min(50, len(pneumonia_test_ds)))),
                                torch.utils.data.Subset(breast_test_ds,   range(min(50, len(breast_test_ds))))])
public_loader = DataLoader(public_pool, batch_size=len(public_pool), shuffle=False)
public_X_fd, _ = next(iter(public_loader))

clients_fd = [FederatedTransferModel().to(DEVICE) for _ in range(2)]
opts_fd    = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
              for m in clients_fd]
global_fd  = FederatedTransferModel().to(DEVICE)
history_feddistill = {'round': [], 'combined_acc': [], 'acc_pneumonia': [], 'acc_breast': []}

for rnd in range(1, CONFIG['num_rounds'] + 1):
    for c, opt, loader in zip(clients_fd, opts_fd, client_loaders):
        train_local(c, loader, opt, CONFIG['local_epochs'], DEVICE)

    client_soft_preds = [get_soft_preds(c, public_X_fd, DEVICE) for c in clients_fd]
    weights = np.array(client_sizes) / sum(client_sizes)
    ensemble_soft = sum(w * p for w, p in zip(weights, client_soft_preds))

    global_opt = optim.Adam(filter(lambda p: p.requires_grad, global_fd.parameters()), lr=CONFIG['learning_rate'])
    T = 4.0
    for _ in range(5):
        global_opt.zero_grad()
        logits = global_fd(public_X_fd.to(DEVICE))
        s_log  = F.log_softmax(logits / T, dim=1)
        t_soft = F.softmax(ensemble_soft.to(DEVICE) / T, dim=1)
        (nn.KLDivLoss(reduction='batchmean')(s_log, t_soft) * (T ** 2)).backward()
        global_opt.step()

    for c in clients_fd:
        c.head.load_state_dict(global_fd.head.state_dict())

    g_acc = evaluate_model(global_fd, test_loader_combined, DEVICE)
    p_acc = evaluate_model(global_fd, client_test_loaders[0], DEVICE)
    b_acc = evaluate_model(global_fd, client_test_loaders[1], DEVICE)
    history_feddistill['round'].append(rnd); history_feddistill['combined_acc'].append(g_acc)
    history_feddistill['acc_pneumonia'].append(p_acc); history_feddistill['acc_breast'].append(b_acc)
    print(f'  Round {rnd:02d} | Combined: {g_acc:.4f} | Pneumonia: {p_acc:.4f} | Breast: {b_acc:.4f}')

print(f'Final combined accuracy: {history_feddistill["combined_acc"][-1]:.4f}')

---
## Section 8: Compare All 4 Algorithms

In [ ]:
results = {
    'FedAvg'          : history_fedavg,
    'FedProx'         : history_fedprox,
    'FedAdam'         : history_fedadam,
    'FedDistillation' : history_feddistill,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Federated Transfer Learning: PneumoniaMNIST + BreastMNIST (2 Binary Medical Clients)',
             fontsize=12, fontweight='bold')

palette = plt.cm.tab10(np.linspace(0, 1, 4))
for (name, hist), color in zip(results.items(), palette):
    axes[0].plot(hist['round'], hist['combined_acc'], '-o', lw=2, ms=5, color=color, label=name)
axes[0].set(xlabel='Round', ylabel='Combined Test Accuracy', title='Accuracy over Rounds')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

final_accs = [hist['combined_acc'][-1] for hist in results.values()]
bars = axes[1].bar(list(results.keys()), final_accs, color=palette)
axes[1].set(ylabel='Final Combined Test Accuracy', title='Final Accuracy Comparison')
axes[1].tick_params(axis='x', rotation=15); axes[1].grid(alpha=0.3, axis='y')
for b, a in zip(bars, final_accs):
    axes[1].text(b.get_x() + b.get_width()/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('federated_transfer_learning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved as federated_transfer_learning_comparison.png')

print('\nFinal ranking (combined test accuracy):')
for name, acc in sorted(zip(results.keys(), final_accs), key=lambda x: -x[1]):
    print(f'   {name:18s}: {acc:.4f}')

print('\nPer-dataset breakdown (final round):')
for name, hist in results.items():
    print(f'   {name:18s}: Pneumonia = {hist["acc_pneumonia"][-1]:.4f} | Breast = {hist["acc_breast"][-1]:.4f}')

---
## Section 9: Summary — What Makes This "Federated Transfer Learning"

**Transfer learning part:** every client starts from the same **ResNet-18 pretrained on ImageNet**. That
backbone is frozen — it never changes, and it's identical on every client and on the server. This is what
lets us skip learning basic visual features (edges, shapes, textures) from scratch on tiny medical datasets,
which normally aren't big enough to train a deep CNN well on their own.

**Federated part:** only a small binary classifier head (a couple of linear layers) sits on top of that
frozen backbone, and that head is what gets trained locally by each client and averaged by the server, round
after round — exactly like FedAvg/FedProx/FedAdam/FedDistillation earlier, just applied to a much smaller
set of trainable parameters.

**Why two totally different medical datasets works here:** PneumoniaMNIST (chest X-rays) and BreastMNIST
(ultrasound) look nothing alike as images, but both are framed as the same binary question — "is this scan
normal/benign (0) or abnormal/malignant (1)?" — so one shared binary head, sitting on top of general-purpose
pretrained features, can plausibly serve both.

**What to tell your professor:**
> *"I implemented Federated Transfer Learning using a frozen, ImageNet-pretrained ResNet-18 backbone shared
> by every client, with only the binary classification head trained federatedly. The two clients hold two
> completely different binary medical datasets — chest X-ray pneumonia detection and breast ultrasound
> tumor classification — unified only by both being 'normal vs abnormal' problems. I compared FedAvg,
> FedProx, FedAdam and FedDistillation on this setup; the per-dataset breakdown shows whether the winning
> algorithm actually helped both clients or mostly favored one."*